<a href="https://colab.research.google.com/github/kolshaan/Hackathon-2026-UpsideDown/blob/Dev/Zenith_Tiger_Case_Study-Deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Agent 1 - The Watchdog**

In [10]:
# df_inventory and df_forecast are already loaded
from openai import OpenAI

from google.colab import userdata
apikey = userdata.get('secret1')
base_url = "https://api.ai-gateway.tigeranalytics.com"

client = OpenAI(
    api_key=apikey,
    base_url=base_url
)

import pandas as pd

# Base raw GitHub URL
base_url = "https://raw.githubusercontent.com/farveznoufal/Hackathon-2026-UpsideDown/main/"

current_inventory = pd.read_csv(base_url + "current_inventory.csv")

inventory_data = current_inventory.to_csv(index=False)

prompt = f"""
ROLE: You are 'The Watchdog', an Inventory Monitor Agent.
PURPOSE: Scans the warehouse network to identify where stock is critically low ("Distress") and where it is overflowing ("Excess").

DATA:
--- INVENTORY ---
{inventory_data}

---

TASK:
1. Identify "Distress" for all SKUs (where On_Hand_Qty < 80% of Safety_Stock_Target).
2. Identify "Excess" for all SKUs (where On_Hand_Qty > 20% greater than Safety_Stock_Target).
3. Pair them up where a location with 'Excess' can supply a location with 'Needs' for all possible combinations.

OUTPUT:
Return ONLY a JSON list of objects with this format:
[{{"SKU": "ID", "Needs": "Loc_A", "Has_Excess": "Loc_B"}}]
"""

response = client.chat.completions.create(
    model="gemini-2.0-flash",
    messages=[
        {"role": "system", "content": "You are a supply chain analyst that only outputs valid JSON."},
        {"role": "user", "content": prompt}
    ],
    response_format={ "type": "json_object" } # This ensures Gemini returns valid JSON
)

watchdog_report = response.choices[0].message.content

## **Check Agent 1 - Output**

In [11]:
import json
import pandas as pd
from google.colab import data_table

# 1. Parse the JSON string into a Python list
report_data = json.loads(watchdog_report)

# 2. Convert to a DataFrame
report_df = pd.DataFrame(report_data)

# 3. Enable Colab's interactive table view
data_table.enable_dataframe_formatter()

# 4. Display the report
report_df

,SKU,Needs,Has_Excess
0,ZEN-101,W01_New_Jersey,W02_Chicago
1,ZEN-301,W07_Seattle,W02_Chicago
2,ZEN-301,W07_Seattle,W03_Atlanta
3,ZEN-301,W07_Seattle,W04_Dallas
4,ZEN-301,W07_Seattle,W05_Denver
5,ZEN-301,W07_Seattle,W06_Los_Angeles
6,ZEN-301,W07_Seattle,W08_Miami
7,ZEN-301,W07_Seattle,W09_St_Louis


### **AGENT 2 - The Economist**


In [12]:
from openai import OpenAI

from google.colab import userdata
apikey = userdata.get('secret1')
base_url = "https://api.ai-gateway.tigeranalytics.com"

client = OpenAI(
    api_key=apikey,
    base_url=base_url
)

import json
import pandas as pd

# Convert dataframes to CSV strings (same pattern as Agent 1)

# Base raw GitHub URL
base_url = "https://raw.githubusercontent.com/farveznoufal/Hackathon-2026-UpsideDown/main/"

# Load CSVs into DataFrames
product_master_data = pd.read_csv(base_url + "product_master.csv")
shipping_matrix_data = pd.read_csv(base_url + "shipping_matrix.csv")
forecast_data = pd.read_csv(base_url + "demand_forecast.csv")

prompt = f"""
ROLE: You are 'The Economist', a Cost Optimization Agent.
PURPOSE: Decide whether it is cheaper to BUY inventory from a vendor or TRANSFER internally.

DATA:
--- IMBALANCE REPORT (from Agent 1) ---
{watchdog_report}

--- PRODUCT MASTER ---
{product_master_data}

--- DEMAND FORECAST ---
{forecast_data}

--- SHIPPING MATRIX ---
{shipping_matrix_data}
---

TASK:
1. For each imbalance:
   - Calculate Cost to BUY = COGS × qty
   - Calculate Cost to TRANSFER = Shipping_Cost_Per_Kg × Quantity
2. Compare Buy vs Transfer.
3. Recommend the cheaper option.
4. Clearly state savings and time advantage.
5. Recommend Transfer Quantity based on DEMAND FORECAST and IMBALANCE REPORT

OUTPUT:
Return ONLY a JSON list with this format:
[
  {{
    "SKU": "ID",
    "Decision": "TRANSFER or BUY",
    "From": "Loc_B",
    "To": "Loc_A",
    "Quantity": 50,
    "Buy_Cost": 500,
    "Transfer_Cost": 100,
    "Savings": 400,
    "Time_Saved_Days": 12
  }}
]
"""

response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {
            "role": "system",
            "content": "You are a supply chain economist. You reason carefully but output ONLY valid JSON."
        },
        {"role": "user", "content": prompt}
    ],
    response_format={"type": "json_object"}
)

economist_raw = response.choices[0].message.content

economist_decisions = json.loads(economist_raw)

final_decisions = []

for decision in economist_decisions:
    sku = decision["SKU"]
    qty = decision["Quantity"]
    origin = decision["From"]
    destination = decision["To"]

    # Buy cost
    product = product_master_data[product_master_data["SKU"] == sku].iloc[0]
    buy_cost = product["COGS"] * qty
    buy_time = product["Factory_Lead_Time_Days"]

    # Transfer cost
    route = shipping_matrix_data[
        (shipping_matrix_data["Origin_Warehouse"] == origin) &
        (shipping_matrix_data["Destination_Warehouse"] == destination)
    ].iloc[0]

    transfer_cost = route["Shipping_Cost_Per_Kg"] * qty
    transfer_time = route["Transit_Time_Days"]

    savings = buy_cost - transfer_cost

    print(
        f"[Economist Monologue] SKU {sku}: "
        f"Buy = ${buy_cost}, Transfer = ${transfer_cost}, "
        f"Savings = ${savings}"
    )

    final_decisions.append({
        "SKU": sku,
        "Decision": "TRANSFER" if savings > 0 else "BUY",
        "From": origin,
        "To": destination,
        "Quantity": qty,
        "Buy_Cost": buy_cost,
        "Transfer_Cost": transfer_cost,
        "Savings": savings,
        "Time_Saved_Days": buy_time - transfer_time
    })

final_decisions



[Economist Monologue] SKU ZEN-101: Buy = $1300.0, Transfer = $62.4, Savings = $1237.6
[Economist Monologue] SKU ZEN-301: Buy = $420.0, Transfer = $36.0, Savings = $384.0
[Economist Monologue] SKU ZEN-301: Buy = $420.0, Transfer = $75.0, Savings = $345.0
[Economist Monologue] SKU ZEN-301: Buy = $420.0, Transfer = $36.0, Savings = $384.0
[Economist Monologue] SKU ZEN-301: Buy = $420.0, Transfer = $15.0, Savings = $405.0
[Economist Monologue] SKU ZEN-301: Buy = $420.0, Transfer = $15.0, Savings = $405.0
[Economist Monologue] SKU ZEN-301: Buy = $420.0, Transfer = $75.0, Savings = $345.0
[Economist Monologue] SKU ZEN-301: Buy = $420.0, Transfer = $36.0, Savings = $384.0


[{'SKU': 'ZEN-101',
  'Decision': 'TRANSFER',
  'From': 'W02_Chicago',
  'To': 'W01_New_Jersey',
  'Quantity': 52,
  'Buy_Cost': np.float64(1300.0),
  'Transfer_Cost': np.float64(62.4),
  'Savings': np.float64(1237.6),
  'Time_Saved_Days': np.int64(4)},
 {'SKU': 'ZEN-301',
  'Decision': 'TRANSFER',
  'From': 'W02_Chicago',
  'To': 'W07_Seattle',
  'Quantity': 30,
  'Buy_Cost': np.float64(420.0),
  'Transfer_Cost': np.float64(36.0),
  'Savings': np.float64(384.0),
  'Time_Saved_Days': np.int64(11)},
 {'SKU': 'ZEN-301',
  'Decision': 'TRANSFER',
  'From': 'W03_Atlanta',
  'To': 'W07_Seattle',
  'Quantity': 30,
  'Buy_Cost': np.float64(420.0),
  'Transfer_Cost': np.float64(75.0),
  'Savings': np.float64(345.0),
  'Time_Saved_Days': np.int64(9)},
 {'SKU': 'ZEN-301',
  'Decision': 'TRANSFER',
  'From': 'W04_Dallas',
  'To': 'W07_Seattle',
  'Quantity': 30,
  'Buy_Cost': np.float64(420.0),
  'Transfer_Cost': np.float64(36.0),
  'Savings': np.float64(384.0),
  'Time_Saved_Days': np.int64(11)}

In [14]:
from google.colab import data_table

df_final = pd.DataFrame(final_decisions)
data_table.enable_dataframe_formatter()
df_final


,SKU,Decision,From,To,Quantity,Buy_Cost,Transfer_Cost,Savings,Time_Saved_Days
0,ZEN-101,TRANSFER,W02_Chicago,W01_New_Jersey,52,1300.0,62.4,1237.6,4
1,ZEN-301,TRANSFER,W02_Chicago,W07_Seattle,30,420.0,36.0,384.0,11
2,ZEN-301,TRANSFER,W03_Atlanta,W07_Seattle,30,420.0,75.0,345.0,9
3,ZEN-301,TRANSFER,W04_Dallas,W07_Seattle,30,420.0,36.0,384.0,11
4,ZEN-301,TRANSFER,W05_Denver,W07_Seattle,30,420.0,15.0,405.0,13
5,ZEN-301,TRANSFER,W06_Los_Angeles,W07_Seattle,30,420.0,15.0,405.0,13
6,ZEN-301,TRANSFER,W08_Miami,W07_Seattle,30,420.0,75.0,345.0,9
7,ZEN-301,TRANSFER,W09_St_Louis,W07_Seattle,30,420.0,36.0,384.0,11


##**Agent 3 - The Executor to generate the final order**

In [15]:
from openai import OpenAI

from google.colab import userdata
apikey = userdata.get('secret1')
base_url = "https://api.ai-gateway.tigeranalytics.com"

client = OpenAI(
    api_key=apikey,
    base_url=base_url
)

import json
import pandas as pd

# Convert dataframes to CSV strings (same pattern as Agent 1)

# Base raw GitHub URL
base_url = "https://raw.githubusercontent.com/farveznoufal/Hackathon-2026-UpsideDown/main/"

executor_input = df_final.to_dict(orient="records")

executor_prompt = f"""
ROLE: You are "The Executor", a Supply Chain Fulfillment AI.

PURPOSE:
Convert approved optimization decisions into a professional Internal Transfer Order (ITO).

INPUT DATA:
--- DECISION RECORDS ---
{json.dumps(executor_input, indent=2)}
---

STRICT INSTRUCTIONS (VERY IMPORTANT):
1. EACH object in the input list represents ONE independent transfer.
2. You MUST copy the following fields EXACTLY AS PROVIDED for each transfer:
   - SKU
   - From
   - To
   - Quantity
   - Savings
   - Time_Saved_Days
3. DO NOT reuse, average, infer, or standardize quantities.
4. DO NOT perform any calculations.
5. DO NOT assume quantities are the same.
6. Preserve row-by-row integrity.

YOUR TASK:
1. Create an Executive Summary using the provided values.
2. Create one Transfer Order per input object.
3. Assign unique Transfer_Order_IDs in sequence (ITO-0001, ITO-0002, ...).
4. Use professional business language suitable for executive approval.

OUTPUT FORMAT:
Return ONLY valid JSON in the following structure:

{{
  "Executive_Summary": {{
    "Total_Transfers": <count>,
    "Total_Savings_USD": <sum of Savings>,
    "Average_Time_Saved_Days": <average>
  }},
  "Transfer_Orders": [
    {{
      "Transfer_Order_ID": "ITO-0001",
      "SKU": "<copied from input>",
      "From_Warehouse": "<copied from input>",
      "To_Warehouse": "<copied from input>",
      "Quantity": <copied from input>,
      "Estimated_Savings_USD": <copied from input>,
      "Estimated_Time_Saved_Days": <copied from input>,
      "Approval_Status": "Pending"
    }}
  ],
  "Business_Notes": "Short executive-friendly justification."
}}

OUTPUT RULES:
- JSON only
- No explanations
- No markdown
- No invented values
"""

response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {
            "role": "system",
            "content": "You are a supply chain executor. You reason carefully but output ONLY valid JSON."
        },
        {"role": "user", "content": prompt}
    ],
    response_format={"type": "json_object"}
)

executor_raw = response.choices[0].message.content
print(executor_raw)


[
  {
    "SKU": "ZEN-101",
    "Decision": "TRANSFER",
    "From": "W02_Chicago",
    "To": "W01_New_Jersey",
    "Quantity": 52,
    "Buy_Cost": 1300.0,
    "Transfer_Cost": 49.92,
    "Savings": 1250.08,
    "Time_Saved_Days": 4
  },
  {
    "SKU": "ZEN-301",
    "Decision": "TRANSFER",
    "From": "W02_Chicago",
    "To": "W07_Seattle",
    "Quantity": 5,
    "Buy_Cost": 70.0,
    "Transfer_Cost": 6.25,
    "Savings": 63.75,
    "Time_Saved_Days": 11
  },
  {
    "SKU": "ZEN-301",
    "Decision": "TRANSFER",
    "From": "W03_Atlanta",
    "To": "W07_Seattle",
    "Quantity": 5,
    "Buy_Cost": 70.0,
    "Transfer_Cost": 12.5,
    "Savings": 57.5,
    "Time_Saved_Days": 11
  },
  {
    "SKU": "ZEN-301",
    "Decision": "TRANSFER",
    "From": "W04_Dallas",
    "To": "W07_Seattle",
    "Quantity": 5,
    "Buy_Cost": 70.0,
    "Transfer_Cost": 10.0,
    "Savings": 60.0,
    "Time_Saved_Days": 11
  },
  {
    "SKU": "ZEN-301",
    "Decision": "TRANSFER",
    "From": "W05_Denver",
    "

In [16]:
import json
import pandas as pd
from google.colab import data_table

executor_output = json.loads(executor_raw)

df_orders = pd.DataFrame(executor_output)
data_table.enable_dataframe_formatter()
df_orders




,SKU,Decision,From,To,Quantity,Buy_Cost,Transfer_Cost,Savings,Time_Saved_Days
0,ZEN-101,TRANSFER,W02_Chicago,W01_New_Jersey,52,1300.0,49.92,1250.08,4
1,ZEN-301,TRANSFER,W02_Chicago,W07_Seattle,5,70.0,6.25,63.75,11
2,ZEN-301,TRANSFER,W03_Atlanta,W07_Seattle,5,70.0,12.50,57.50,11
3,ZEN-301,TRANSFER,W04_Dallas,W07_Seattle,5,70.0,10.00,60.00,11
4,ZEN-301,TRANSFER,W05_Denver,W07_Seattle,5,70.0,15.00,55.00,11
5,ZEN-301,TRANSFER,W06_Los_Angeles,W07_Seattle,5,70.0,7.50,62.50,11
6,ZEN-301,TRANSFER,W08_Miami,W07_Seattle,5,70.0,12.50,57.50,11
7,ZEN-301,TRANSFER,W09_St_Louis,W07_Seattle,5,70.0,15.00,55.00,11
